# Notebook 02 : Measurement Framework
### Feature Engineering · Joins · Incrementality · Automated Alerting

This notebook ingests the raw video CSVs and builds the master analytics table used for all downstream analysis and outputs.

**Outputs written to `data/modeled/`:**
- `creator_campaign_metrics.csv` : 21-column master table (14 rows)
- `incrementality_q3_q4.csv` : Q3→Q4 delta per creator
- `flagging_alerts.csv` : automated performance alerts

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

DATA_RAW     = '../data/raw'
DATA_MODELED = '../data/modeled'

# Fixed analysis date for reproducibility
ANALYSIS_DATE = datetime(2025, 1, 15)

# Estimated monthly spend (Influencer Marketing Hub 2024 benchmarks)
MONTHLY_SPEND = {
    'TenZ':           25000,
    'tarik':          20000,
    'Kyedae':         22000,
    'aceu':           28000,
    'Sinatraa':        8000,
    'Protatomonster':  7000,
}

# Benchmarks
ER_BENCHMARK  = 3.87  # Statista 2024 gaming YouTube average
CPE_EFFICIENT = 0.11
CPE_ACCEPTABLE = 0.25

print('Config loaded. ANALYSIS_DATE:', ANALYSIS_DATE.date())

## 1. Load Raw Data

In [ ]:
df_q3 = pd.read_csv(f'{DATA_RAW}/q3_2024_videos.csv', parse_dates=['published_date'])
df_q4 = pd.read_csv(f'{DATA_RAW}/q4_2024_videos.csv', parse_dates=['published_date'])
df_ch = pd.read_csv(f'{DATA_RAW}/channel_stats.csv')
df_tr = pd.read_csv(f'{DATA_RAW}/search_interest_monthly.csv')

print(f'Q3: {len(df_q3)} rows | Q4: {len(df_q4)} rows')
print(f'Channels: {len(df_ch)} | Trends months: {len(df_tr)}')

# Combine quarters
df_q3['quarter'] = 'Q3'
df_q4['quarter'] = 'Q4'
df_all = pd.concat([df_q3, df_q4], ignore_index=True)
print(f'Combined: {len(df_all)} rows')

## 2. Feature Engineering

### 2a. ISO 8601 Duration Parsing

In [ ]:
def parse_duration(iso):
    """Convert PT#H#M#S to total seconds."""
    if pd.isna(iso):
        return 0
    h = int(re.search(r'(\d+)H', iso).group(1)) if 'H' in iso else 0
    m = int(re.search(r'(\d+)M', iso).group(1)) if 'M' in iso else 0
    s = int(re.search(r'(\d+)S', iso).group(1)) if 'S' in iso else 0
    return h*3600 + m*60 + s

def content_type(secs):
    if secs < 120:   return 'Short'
    if secs < 1200:  return 'Mid-form'
    return 'Long-form'

df_all['duration_sec']  = df_all['duration'].apply(parse_duration)
df_all['content_type']  = df_all['duration_sec'].apply(content_type)

# Days since publish (normalized velocity)
df_all['published_date'] = pd.to_datetime(df_all['published_date'])
df_all['days_since']     = (ANALYSIS_DATE - df_all['published_date']).dt.days.clip(lower=1)
df_all['views_per_day']  = (df_all['views'] / df_all['days_since']).round(1)

print('Content type distribution:')
print(df_all['content_type'].value_counts())

### 2b. Engagement & Quality Metrics

In [ ]:
# Ensure engagements column
if 'engagements' not in df_all.columns:
    df_all['engagements'] = df_all['likes'] + df_all['comments']

# Quality Engagement Index: comments weighted 3x (intent signal)
df_all['qei'] = ((df_all['likes'] + df_all['comments'] * 3)
                 / df_all['views'].replace(0, 1) * 100).round(4)

# Reach Index
df_all['er'] = (df_all['engagements'] / df_all['views'].replace(0, 1) * 100).round(4)
df_all['reach_index'] = (df_all['views'] * (1 + df_all['er'] / 100)).round(0)

# Watch time (modelled at 50% completion)
df_all['watch_time_hrs'] = (df_all['views'] * df_all['duration_sec'] * 0.5 / 3600).round(1)

# TenZ viral outlier flag
df_all['is_viral_outlier'] = (
    (df_all['creator'] == 'TenZ') &
    (df_all['quarter'] == 'Q3') &
    (df_all['views'] > 5_000_000)
)

print(f"Viral outlier rows flagged: {df_all['is_viral_outlier'].sum()}")
print(df_all[df_all['is_viral_outlier']][['creator','title','views','er']])

## 3. Three Multi-Source Joins

```
Video data  ──JOIN──▶  channel_stats        (add subscribers, tier)
   │
   ▼
Monthly agg ──JOIN──▶  search_interest      (add Google Trends context)
   │
   ▼
Q4 monthly  ──JOIN──▶  Q3 monthly           (compute ΔER incrementality)
```

In [ ]:
# JOIN 1: Add channel tier from channel_stats
df_all = df_all.merge(
    df_ch[['creator', 'tier']],
    on='creator', how='left'
)

# Monthly aggregation (exclude viral outlier from ER calc)
df_clean = df_all[~df_all['is_viral_outlier']].copy()
df_all['month'] = df_all['published_date'].dt.to_period('M').astype(str)
df_clean['month'] = df_clean['published_date'].dt.to_period('M').astype(str)

monthly_agg = df_clean.groupby(['creator', 'month', 'tier']).agg(
    video_count    = ('video_id',      'count'),
    total_views    = ('views',         'sum'),
    total_likes    = ('likes',         'sum'),
    total_comments = ('comments',      'sum'),
    total_eng      = ('engagements',   'sum'),
    avg_duration   = ('duration_sec',  'mean'),
    avg_vpd        = ('views_per_day', 'mean'),
    total_watch_hrs= ('watch_time_hrs','sum'),
    avg_reach_idx  = ('reach_index',   'mean'),
    avg_qei        = ('qei',           'mean'),
).reset_index()

monthly_agg['er'] = (monthly_agg['total_eng'] / monthly_agg['total_views'].replace(0,1) * 100).round(4)
monthly_agg['est_monthly_spend'] = monthly_agg['creator'].map(MONTHLY_SPEND)
monthly_agg['cpe'] = (monthly_agg['est_monthly_spend'] / monthly_agg['total_eng'].replace(0,1)).round(4)
monthly_agg['cpe_tier'] = monthly_agg['cpe'].apply(
    lambda x: 'Efficient' if x <= CPE_EFFICIENT else ('Acceptable' if x <= CPE_ACCEPTABLE else 'Review')
)

# JOIN 2: Add Google Trends
monthly_agg = monthly_agg.merge(df_tr, on='month', how='left')
monthly_agg.rename(columns={'avg_interest': 'avg_search_interest'}, inplace=True)

print(f'Monthly aggregation: {len(monthly_agg)} rows')
print(monthly_agg[['creator','month','total_views','er','cpe','cpe_tier']].to_string(index=False))

## 4. Incrementality — Q3 → Q4 Delta (JOIN 3)

In [ ]:
monthly_agg['quarter'] = monthly_agg['month'].apply(
    lambda m: 'Q3' if m in ['2024-07','2024-08','2024-09'] else 'Q4'
)

# Q3 and Q4 averages per creator
q3_avg = monthly_agg[monthly_agg['quarter']=='Q3'].groupby('creator').agg(
    q3_avg_er    = ('er',          'mean'),
    q3_avg_views = ('total_views', 'mean'),
    q3_video_ct  = ('video_count', 'sum'),
).reset_index()

q4_avg = monthly_agg[monthly_agg['quarter']=='Q4'].groupby('creator').agg(
    q4_avg_er    = ('er',          'mean'),
    q4_avg_views = ('total_views', 'mean'),
    q4_video_ct  = ('video_count', 'sum'),
).reset_index()

# JOIN 3: Q4 ← Q3
inc = q4_avg.merge(q3_avg, on='creator', how='left')
inc['delta_er_pp']     = (inc['q4_avg_er'] - inc['q3_avg_er']).round(4)
inc['delta_views_pct'] = ((inc['q4_avg_views'] - inc['q3_avg_views']) / inc['q3_avg_views'].replace(0,1) * 100).round(1)
inc['trend'] = inc['delta_er_pp'].apply(
    lambda d: 'Improving' if d > 0 else ('Declining' if d < -0.3 else 'Stable')
)

# Save incrementality
cols_out = ['creator','q3_avg_er','q3_avg_views','q3_video_ct',
            'q4_avg_er','q4_avg_views','q4_video_ct','delta_er_pp','delta_views_pct','trend']
inc[cols_out].round(4).to_csv(f'{DATA_MODELED}/incrementality_q3_q4.csv', index=False)
print('Incrementality saved.')
print(inc[['creator','q3_avg_er','q4_avg_er','delta_er_pp','trend']].to_string(index=False))

## 5. Add Trend to Master Table + Save

In [ ]:
# Add delta_er and trend to monthly table
monthly_agg = monthly_agg.merge(
    inc[['creator','delta_er_pp','trend']],
    on='creator', how='left'
)

# Final column order
final_cols = [
    'creator','month','quarter','tier','video_count',
    'total_views','total_eng','er','avg_qei','avg_reach_idx',
    'avg_vpd','avg_duration','total_watch_hrs',
    'est_monthly_spend','cpe','cpe_tier',
    'avg_search_interest','delta_er_pp','trend'
]
master = monthly_agg[[c for c in final_cols if c in monthly_agg.columns]]
master.to_csv(f'{DATA_MODELED}/creator_campaign_metrics.csv', index=False)
print(f'Master table saved: {len(master)} rows × {len(master.columns)} cols')
master.head()

## 6. Automated Alerting System

In [ ]:
alerts = []

for _, row in master.iterrows():
    c, m = row['creator'], row['month']
    
    # Alert 1: CPE Overrun
    if row['cpe'] > CPE_ACCEPTABLE:
        alerts.append({'creator': c, 'month': m,
                       'alert_type': 'CPE Overrun',
                       'detail': f"CPE ${row['cpe']:.2f} exceeds ${CPE_ACCEPTABLE} ceiling"})
    
    # Alert 2: Declining ER (use delta_er for Q4 months only)
    if row['quarter'] == 'Q4' and row.get('delta_er_pp', 0) < -0.5:
        alerts.append({'creator': c, 'month': m,
                       'alert_type': 'Declining ER',
                       'detail': f"ΔER {row['delta_er_pp']:+.2f}pp vs Q3 avg"})
    
    # Alert 3: High Volume / Low Quality
    if row['total_views'] > 500_000 and row['er'] < 2.0:
        alerts.append({'creator': c, 'month': m,
                       'alert_type': 'High Volume / Low Quality',
                       'detail': f"{row['total_views']:,.0f} views but ER {row['er']:.2f}%"})
    
    # Alert 4: Below Platform Average
    if row['er'] < ER_BENCHMARK * 0.5:
        alerts.append({'creator': c, 'month': m,
                       'alert_type': 'Below Platform Average',
                       'detail': f"ER {row['er']:.2f}% < 50% of {ER_BENCHMARK}% benchmark"})

df_alerts = pd.DataFrame(alerts)
df_alerts.to_csv(f'{DATA_MODELED}/flagging_alerts.csv', index=False)
print(f'{len(df_alerts)} alerts fired.')
print(df_alerts['alert_type'].value_counts())

## 7. Q4 Leaderboard Summary

In [ ]:
q4_summary = master[master['quarter']=='Q4'].groupby('creator').agg(
    total_views = ('total_views', 'sum'),
    total_eng   = ('total_eng',   'sum'),
    avg_er      = ('er',          'mean'),
    avg_cpe     = ('cpe',         'mean'),
    est_spend   = ('est_monthly_spend', 'mean'),
    delta_er    = ('delta_er_pp', 'first'),
    trend       = ('trend',       'first'),
).reset_index().sort_values('avg_er', ascending=False)

q4_summary['avg_er']  = q4_summary['avg_er'].round(2)
q4_summary['avg_cpe'] = q4_summary['avg_cpe'].round(2)
q4_summary['delta_er']= q4_summary['delta_er'].round(2)

print('=== Q4 2024 CREATOR LEADERBOARD (by ER) ===')
print(q4_summary[['creator','avg_er','avg_cpe','total_views','delta_er','trend']].to_string(index=False))